# Yusen: Least Squares and Ridge on Shu's baseline

**Goal:** reproduce Shu's saved Ridge result, then compare the shared Least Squares and Ridge implementations on the same matrices after removing exact redundant columns.

This experiment preserves Shu's feature choices and preprocessing order. It does not tune features, lambda, or the classification threshold. A separate, explicitly labeled comparison removes two dependent columns from both models because Least Squares cannot uniquely fit the original design matrix. All rows of the labeled dataset participate in the 75%/25% split; only a subset of feature columns enters the models. The unlabeled competition test set is not needed here.

Run all cells from a fresh kernel. NumPy is the only numerical dependency. Loading the full dataset requires several GB of available RAM. The notebook finds the repository when started from its root, `experiments/`, or the parent workspace.

In [1]:
import ast
import csv
import hashlib
import json
import platform
import sys
from itertools import islice
from pathlib import Path

import numpy as np

# Resolve paths independently of the notebook kernel's starting directory.
candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / "cs433-project1"]
ROOT = next((p for p in candidates
             if (p / "implementations.py").is_file()
             and (p / "experiments/Shu.run.ipynb").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Start the kernel inside cs433-project1 or its parent workspace.")
sys.path.insert(0, str(ROOT))

from implementations import least_squares, ridge_regression
from preprocessing.preprocess import train_val_split
from preprocessing.shu_baseline_config import (
    ORDERED_NAMES, DECODE_ORDERED_NAMES, CATEGORICAL_NAMES,
    DROP_CANDIDATE_NAMES, ORDERED_RULES,
)

SEED = 42
VAL_RATIO = 0.25
LAMBDA = 0.001
THRESHOLD = -0.55
DATA = ROOT / "dataset"
print("Repository:", ROOT.name)
print("Python:", platform.python_version(), "NumPy:", np.__version__)
print(f"seed={SEED}, validation={VAL_RATIO}, lambda={LAMBDA}, threshold={THRESHOLD}")

Repository: cs433-project1
Python: 3.14.6 NumPy: 2.4.6
seed=42, validation=0.25, lambda=0.001, threshold=-0.55


## 1. Check the provenance of the feature rules

The Python configuration is a literal snapshot of Shu's feature groups and ordered-feature rules. We compare the values with his notebook before running. This reads notebook JSON and literal assignments; it does not execute Shu's notebook. If the source rules change later, review and update the snapshot explicitly.

In [2]:
shu_path = ROOT / "experiments/Shu.run.ipynb"
shu_notebook = json.loads(shu_path.read_text(encoding="utf-8"))
config = {
    "ordered_names": ORDERED_NAMES,
    "decode_ordered_names": DECODE_ORDERED_NAMES,
    "categorical_names": CATEGORICAL_NAMES,
    "drop_candidate_names": DROP_CANDIDATE_NAMES,
    "ordered_rules": ORDERED_RULES,
}
source_config = {}
for cell in shu_notebook["cells"]:
    if cell["cell_type"] != "code":
        continue
    for node in ast.parse("".join(cell["source"])).body:
        if isinstance(node, ast.Assign) and isinstance(node.targets[0], ast.Name):
            name = node.targets[0].id
            if name in config:
                source_config[name] = ast.literal_eval(node.value)
assert source_config == config, "Shu's configuration changed; review the snapshot."
print("All five configuration objects match Shu's notebook.")

All five configuration objects match Shu's notebook.


## 2. Load labeled data and reproduce the split

Load only `x_train.csv` and `y_train.csv`. Read feature rows in chunks to limit CSV parsing memory, preserving row order, float64 values, and blank-to-NaN handling. Check sample IDs before discarding them. The existing `train_val_split` helper reproduces Shu's random split exactly.

In [3]:
labels = np.loadtxt(DATA / "y_train.csv", delimiter=",", skiprows=1, dtype=int)
train_ids, y_all = labels[:, 0], labels[:, 1]
assert set(np.unique(y_all)) == {-1, 1}

with (DATA / "x_train.csv").open(encoding="utf-8", newline="") as stream:
    header = next(csv.reader(stream))
    feature_names = np.array(header[1:])  # Exclude the sample ID column.
    x_all = np.empty((len(y_all), len(feature_names)), dtype=float)
    offset = 0
    while True:
        lines = list(islice(stream, 10000))
        if not lines:
            break
        block = np.genfromtxt(lines, delimiter=",", dtype=float, ndmin=2)
        stop = offset + len(block)
        assert stop <= len(y_all), "More feature rows than labels."
        np.testing.assert_array_equal(block[:, 0], train_ids[offset:stop])
        x_all[offset:stop] = block[:, 1:]
        offset = stop
    assert offset == len(y_all), "Feature/label row count mismatch."
del block, lines

x_train, x_val, y_train, y_val = train_val_split(
    x_all, y_all, val_ratio=VAL_RATIO, seed=SEED
)
print("Original features:", len(feature_names))
print("Train:", x_train.shape, "Validation:", x_val.shape)
print("Positive fraction - train:", np.mean(y_train == 1),
      "validation:", np.mean(y_val == 1))
del x_all, y_all, labels, train_ids

Original features: 321
Train: (246102, 321) Validation: (82033, 321)
Positive fraction - train: 0.08832516598808623 validation: 0.08823278436726684


## 3. Select the same feature groups

Count distinct nonmissing values **on the training split only**. Keep Shu's one-observed-value columns, replacing missing entries with -1, and his 111 ordered/numeric columns. The other groups are classified but not used in this baseline; they are not all proven irrelevant.

In [4]:
unique_count = np.array([
    np.unique(col[~np.isnan(col)]).size for col in x_train.T
])
one_value_idx = np.flatnonzero(unique_count == 1)
name_to_idx = {name: i for i, name in enumerate(feature_names)}

def group_indices(names):
    return np.array([name_to_idx[name] for name in names
                     if name in name_to_idx and unique_count[name_to_idx[name]] > 1],
                    dtype=int)

ordered_idx = group_indices(ORDERED_NAMES)
assert list(feature_names[ordered_idx]) == ORDERED_NAMES
classified = np.concatenate([
    one_value_idx, ordered_idx, group_indices(DECODE_ORDERED_NAMES),
    group_indices(CATEGORICAL_NAMES), group_indices(DROP_CANDIDATE_NAMES),
])
assert len(classified) == len(np.unique(classified)) == len(feature_names)

x_one_train = x_train[:, one_value_idx].copy()
x_one_val = x_val[:, one_value_idx].copy()
x_one_train[np.isnan(x_one_train)] = -1
x_one_val[np.isnan(x_one_val)] = -1
x_ord_train = x_train[:, ordered_idx].copy()
x_ord_val = x_val[:, ordered_idx].copy()
print("One-observed-value features:", list(feature_names[one_value_idx]))
print("Ordered/numeric features:", len(ordered_idx))
# Release unused raw columns before allocating model matrices.
del x_train, x_val

One-observed-value features: [np.str_('CTELENUM'), np.str_('COLGHOUS'), np.str_('STATERES'), np.str_('CTELNUM1'), np.str_('CELLFON2'), np.str_('CCLGHOUS')]
Ordered/numeric features: 111


## 4. Reproduce ordered-feature preprocessing

Apply the same special-code rules in the same order as Shu. Estimate imputation means, centering means, and population standard deviations on the training split, then reuse them on validation. Stop on all-missing or zero-variance columns instead of silently changing the baseline.

In [5]:
def apply_ordered_rules(x):
    """Apply Shu's code mappings without changing the input array."""
    result = x.copy()
    for j, name in enumerate(ORDERED_NAMES):
        rule = ORDERED_RULES.get(name)
        if rule is None:
            continue
        for old_code, new_code in rule["remap_codes"].items():
            result[result[:, j] == old_code, j] = new_code
        for value in rule["zero_codes"]:
            result[result[:, j] == value, j] = 0
        for value in rule["nan_codes"]:
            result[result[:, j] == value, j] = np.nan
    return result

x_ord_train = apply_ordered_rules(x_ord_train)
x_ord_val = apply_ordered_rules(x_ord_val)
assert not np.any(np.all(np.isnan(x_ord_train), axis=0)), "All-missing training feature."
impute_mean = np.nanmean(x_ord_train, axis=0)
x_ord_train = np.where(np.isnan(x_ord_train), impute_mean, x_ord_train)
x_ord_val = np.where(np.isnan(x_ord_val), impute_mean, x_ord_val)

center = np.mean(x_ord_train, axis=0)
scale = np.std(x_ord_train, axis=0)
assert np.all(scale > 0), "Zero-variance ordered feature; review before changing the baseline."
x_ord_train = (x_ord_train - center) / scale
x_ord_val = (x_ord_val - center) / scale

# Preserve Shu's feature order and add exactly one intercept column.
tx_train = np.c_[np.ones(len(y_train)), x_one_train, x_ord_train]
tx_val = np.c_[np.ones(len(y_val)), x_one_val, x_ord_val]
model_feature_names = ["intercept", *feature_names[one_value_idx], *ORDERED_NAMES]
assert np.isfinite(tx_train).all() and np.isfinite(tx_val).all()
print("Model matrix - train:", tx_train.shape, "validation:", tx_val.shape)
del x_one_train, x_one_val, x_ord_train, x_ord_val

Model matrix - train: (246102, 118) validation: (82033, 118)


## 5. Define one evaluation procedure

Both methods produce continuous scores, not probabilities. Use Shu's fixed threshold -0.55 for both; positive predictions satisfy `score >= threshold`. Report positive-class precision, recall and F1, plus accuracy, confusion counts, and half-MSE. Returned training loss is not a classification metric.

In [6]:
def evaluate(y, scores, threshold=THRESHOLD):
    """Evaluate the positive class (+1), with safe zero-denominator handling."""
    predicted = np.where(scores >= threshold, 1, -1)
    tp = int(np.sum((predicted == 1) & (y == 1)))
    tn = int(np.sum((predicted == -1) & (y == -1)))
    fp = int(np.sum((predicted == 1) & (y == -1)))
    fn = int(np.sum((predicted == -1) & (y == 1)))
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    return {
        "tp": tp, "tn": tn, "fp": fp, "fn": fn,
        "accuracy": (tp + tn) / len(y), "precision": precision, "recall": recall,
        "f1": 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0,
        "half_mse": float(0.5 * np.mean((y - scores) ** 2)),
    }

# Small known confusion matrix: one TP, TN, FP and FN.
check = evaluate(np.array([1, -1, 1, -1]), np.array([1., -1., -1., 1.]))
assert [check[k] for k in ["tp", "tn", "fp", "fn"]] == [1, 1, 1, 1]
assert check["f1"] == check["accuracy"] == 0.5

## 6. Reproduce Shu's Ridge and verify the shared implementation

First use Shu's exact normal-equation expression as a reference. Then call the shared `ridge_regression` and verify matching weights and validation scores. The historical confusion counts below come from the saved output in `Shu.run.ipynb`. A mismatch stops the notebook for investigation rather than claiming reproduction.

In [7]:
# Reference only: this is the formula in Shu.run.ipynb, not a new model variant.
n, d = tx_train.shape
shu_w = np.linalg.solve(
    tx_train.T @ tx_train + 2 * n * LAMBDA * np.eye(d),
    tx_train.T @ y_train,
)
shu_scores = tx_val @ shu_w
shu_metrics = evaluate(y_val, shu_scores)
expected_counts = {"tp": 2615, "tn": 69418, "fp": 5377, "fn": 4623}
assert {key: shu_metrics[key] for key in expected_counts} == expected_counts

w_ridge, loss_ridge = ridge_regression(y_train, tx_train, LAMBDA)
original_ridge_scores = tx_val @ w_ridge
np.testing.assert_allclose(w_ridge, shu_w, rtol=1e-10, atol=1e-12)
np.testing.assert_allclose(original_ridge_scores, shu_scores, rtol=1e-10, atol=1e-12)
np.testing.assert_allclose(loss_ridge, 0.5 * np.mean((y_train - tx_train @ w_ridge)**2))
print("Historical Ridge confusion counts reproduced:", expected_counts)
print("Shared Ridge matches Shu's weights and scores.")
print("Ridge validation metrics:", evaluate(y_val, original_ridge_scores))
original_ridge_loss = float(loss_ridge)

Historical Ridge confusion counts reproduced: {'tp': 2615, 'tn': 69418, 'fp': 5377, 'fn': 4623}
Shared Ridge matches Shu's weights and scores.
Ridge validation metrics: {'tp': 2615, 'tn': 69418, 'fp': 5377, 'fn': 4623, 'accuracy': 0.8780978386746797, 'precision': 0.3272022022022022, 'recall': 0.3612876485216911, 'f1': 0.3434011818778726, 'half_mse': 0.14262965711309283}


## 7. Remove exact redundant columns for a fair, solvable comparison

The original training matrix has rank 116 but 118 columns. After Shu's missing-value replacement, `CTELNUM1` and `CELLFON2` are both exactly the negative of `CTELENUM`. These columns do not add independent information.

Choose columns to keep **using training data only**, preserving the first member of each identical/sign-reversed pair in the one-observed-value group. Apply that same column mask to validation data and to both shared models. Verify full column rank before solving Least Squares. No pseudoinverse or hidden regularization is used.

**Important:** removing redundant columns preserves the possible linear predictions but changes Ridge's L2 penalty on the coefficient representation. Therefore retain the original 118-column Ridge result separately, and compare Least Squares with a newly fitted Ridge on the same 116-column matrices.

In [8]:
original_singular_values = np.linalg.svd(tx_train, compute_uv=False)
original_tolerance = original_singular_values[0] * max(tx_train.shape) * np.finfo(float).eps
original_rank = int(np.sum(original_singular_values > original_tolerance))
print("Original training matrix rank:", original_rank, "of", tx_train.shape[1])

# Only these six special columns are candidates for exact duplicate removal.
# The decision is made from training values; validation never selects features.
keep_columns = np.ones(tx_train.shape[1], dtype=bool)
representatives = []
removed_columns = []
for j in range(1, 1 + len(one_value_idx)):
    for previous in representatives:
        same = np.array_equal(tx_train[:, j], tx_train[:, previous])
        opposite = np.array_equal(tx_train[:, j], -tx_train[:, previous])
        if same or opposite:
            keep_columns[j] = False
            removed_columns.append({"feature": model_feature_names[j],
                                    "representative": model_feature_names[previous],
                                    "sign": 1 if same else -1})
            break
    else:
        representatives.append(j)

tx_train_compare = tx_train[:, keep_columns]
tx_val_compare = tx_val[:, keep_columns]
comparison_feature_names = list(np.array(model_feature_names)[keep_columns])
print("Removed redundant columns:", removed_columns)
singular_values = np.linalg.svd(tx_train_compare, compute_uv=False)
rank_tolerance = singular_values[0] * max(tx_train_compare.shape) * np.finfo(float).eps
matrix_rank = int(np.sum(singular_values > rank_tolerance))
print("Comparison training matrix rank:", matrix_rank, "of", tx_train_compare.shape[1])
print("Comparison matrix condition number:", singular_values[0] / singular_values[-1])
assert matrix_rank == tx_train_compare.shape[1], "Further dependencies require review."

w_ls, loss_ls = least_squares(y_train, tx_train_compare)
w_ridge_compare, loss_ridge_compare = ridge_regression(y_train, tx_train_compare, LAMBDA)
ls_scores = tx_val_compare @ w_ls
ridge_scores = tx_val_compare @ w_ridge_compare
assert np.isfinite(w_ls).all() and np.isfinite(ls_scores).all()
assert np.isfinite(w_ridge_compare).all() and np.isfinite(ridge_scores).all()
np.testing.assert_allclose(loss_ls, 0.5 * np.mean((y_train - tx_train_compare @ w_ls)**2))
np.testing.assert_allclose(loss_ridge_compare,
                           0.5 * np.mean((y_train - tx_train_compare @ w_ridge_compare)**2))
# The gradient should vanish at the Least Squares optimum.
np.testing.assert_allclose(tx_train_compare.T @ (tx_train_compare @ w_ls - y_train) / n,
                           0.0, atol=1e-9)
print("Least Squares validation metrics:", evaluate(y_val, ls_scores))
print("Comparable Ridge validation metrics:", evaluate(y_val, ridge_scores))

Original training matrix rank: 116 of 118
Removed redundant columns: [{'feature': np.str_('CTELNUM1'), 'representative': np.str_('CTELENUM'), 'sign': -1}, {'feature': np.str_('CELLFON2'), 'representative': np.str_('CTELENUM'), 'sign': -1}]
Comparison training matrix rank: 116 of 116
Comparison matrix condition number: 894.1322174344361
Least Squares validation metrics: {'tp': 2650, 'tn': 69303, 'fp': 5492, 'fn': 4588, 'accuracy': 0.8771226213840772, 'precision': 0.325472856791943, 'recall': 0.366123238463664, 'f1': 0.34460338101430427, 'half_mse': 0.14242511677974115}
Comparable Ridge validation metrics: {'tp': 2615, 'tn': 69418, 'fp': 5377, 'fn': 4623, 'accuracy': 0.8780978386746797, 'precision': 0.3272022022022022, 'recall': 0.3612876485216911, 'f1': 0.3434011818778726, 'half_mse': 0.14262965942172134}


## 8. Compare and save the experiment summary

The always-negative predictor contextualizes accuracy under class imbalance. The comparison uses a fixed threshold inherited from Shu, not each model's best tuned threshold. A single split does not establish a robust winner. No test-set prediction or leaderboard score is produced here.

In [9]:
results = [
    {"model": "Shu Ridge reference", "columns": d, "lambda": LAMBDA,
     "train_half_mse": float(0.5 * np.mean((y_train - tx_train @ shu_w)**2)),
     **shu_metrics},
    {"model": "Shared Ridge original", "columns": d, "lambda": LAMBDA,
     "train_half_mse": original_ridge_loss, **evaluate(y_val, original_ridge_scores)},
    {"model": "Shared Ridge reduced", "columns": tx_train_compare.shape[1], "lambda": LAMBDA,
     "train_half_mse": float(loss_ridge_compare), **evaluate(y_val, ridge_scores)},
    {"model": "Least Squares reduced", "columns": tx_train_compare.shape[1], "lambda": None,
     "train_half_mse": float(loss_ls), **evaluate(y_val, ls_scores)},
    {"model": "Always negative", "columns": 0, "lambda": None,
     "train_half_mse": float(0.5 * np.mean((y_train + 1)**2)),
     **evaluate(y_val, -np.ones(len(y_val)))},
]
print(f'{"Model":24s} {"Train loss":>11s} {"Val loss":>11s} {"Accuracy":>10s} '
      f'{"Precision":>10s} {"Recall":>10s} {"F1":>10s}')
for row in results:
    print(f'{row["model"]:24s} {row["train_half_mse"]:11.6f} {row["half_mse"]:11.6f} '
          f'{row["accuracy"]:10.6f} {row["precision"]:10.6f} '
          f'{row["recall"]:10.6f} {row["f1"]:10.6f}')

def file_sha256(path):
    """Record input provenance without copying raw data into the results."""
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

summary = {
    "settings": {"seed": SEED, "validation_ratio": VAL_RATIO,
                 "lambda": LAMBDA, "threshold": THRESHOLD},
    "environment": {"python": platform.python_version(), "numpy": np.__version__},
    "train_rows": len(y_train), "validation_rows": len(y_val),
    "raw_features": len(feature_names), "model_columns_including_intercept": d,
    "original_matrix_rank": original_rank, "feature_names": model_feature_names,
    "comparison_matrix_rank": matrix_rank,
    "comparison_columns_including_intercept": tx_train_compare.shape[1],
    "comparison_feature_names": comparison_feature_names,
    "removed_columns": removed_columns,
    "historical_ridge_reproduced": True, "shared_ridge_matches_reference": True,
    "sha256": {str(p.relative_to(ROOT)): file_sha256(p) for p in [
        DATA / "x_train.csv", DATA / "y_train.csv", shu_path,
        ROOT / "implementations.py", ROOT / "preprocessing/shu_baseline_config.py",
        ROOT / "preprocessing/preprocess.py",
    ]},
    "results": results,
}
output_path = ROOT / "experiments/yusen_baseline_results.json"
output_path.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
print("Saved:", output_path.relative_to(ROOT))

Model                     Train loss    Val loss   Accuracy  Precision     Recall         F1
Shu Ridge reference         0.142412    0.142630   0.878098   0.327202   0.361288   0.343401
Shared Ridge original       0.142412    0.142630   0.878098   0.327202   0.361288   0.343401
Shared Ridge reduced        0.142412    0.142630   0.878098   0.327202   0.361288   0.343401
Least Squares reduced       0.142218    0.142425   0.877123   0.325473   0.366123   0.344603
Always negative             0.176650    0.176466   0.911767   0.000000   0.000000   0.000000
Saved: experiments\yusen_baseline_results.json


## Next experiment

Keep this notebook as the reproducible baseline. Change one feature group or preprocessing choice at a time in a separate experiment, using the same split. Tune regularization and classification thresholds on validation data only; repeated tuning can overfit this split, so use further validation before final claims. The original Shu notebooks and shared model implementations remain separate from this experiment.